# ARD — Colab 부트스트랩
Colab **Pro** GPU 런타임(L4/A100)에서 실행. 순서대로 셀 실행.

사전: Colab Secret에 `HF_TOKEN` (gated 모델 접근용) 등록.

In [ ]:
# 1) 저장소 + 의존성
REPO = 'https://github.com/theman001/LLM-Jailbreak.git'
!git clone $REPO ard 2>/dev/null || (cd ard && git pull)
%cd ard
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2) Drive 마운트 + HF 토큰 + 데이터 경로
import os
from google.colab import drive, userdata
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/ARD'
os.makedirs(DATA_DIR, exist_ok=True)
os.environ['ARD_DATA_DIR'] = DATA_DIR              # app.py가 읽음
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('data dir:', DATA_DIR)
print('pairs.csv 있음?', os.path.exists(f'{DATA_DIR}/pairs.csv'))

In [ ]:
# 3) Streamlit 기동 + cloudflared 터널 (URL 출력 대기)
import subprocess, time, re, urllib.request, os
BIN = '/tmp/cloudflared'
if not os.path.exists(BIN):
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', BIN)
    os.chmod(BIN, 0o755)
subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.headless', 'true'])
time.sleep(8)
log = open('/tmp/cf.log', 'w+')
subprocess.Popen([BIN, 'tunnel', '--url', 'http://localhost:8501'], stdout=log, stderr=log)
for _ in range(30):
    time.sleep(2); log.seek(0); m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', log.read())
    if m: print('OPEN:', m.group(0)); break
else:
    print('터널 URL을 못 찾음 — /tmp/cf.log 확인')